In [53]:
import numpy as np
from mealpy import SMA, GA, BRO, FloatVar

In [54]:
def rosenbrock(x):
    s = sum([100*(x[i+1]-x[i]**2)**2 + (1-x[i])**2 for i in range(len(x)-1)])
    return s

def mishra(x):
    return np.sin(x[1])*np.exp((1-np.cos(x[0]))**2) + np.cos(x[0])*np.exp((1-np.sin(x[1]))**2) + (x[0]-x[1])**2

In [55]:
def multi_obj(solution):
    def g1(x): #constraint for rosenbrock
        return x[0]**2 + x[1]**2 - 2
    def g2(x): #constraint for mishra
        return (x[0]+5)**2 + (x[1]+5)**2 - 25
    def violate1(x): 
        return 0 if x <= 0 else x
    def violate2(x):
        return 0 if x < 0 else x
    f1 = rosenbrock(solution)
    f2 = mishra(solution)

    f1 += violate1(g1(solution))**2
    f2 += violate2(g2(solution))**2

    return [f1, f2]

def single_obj_r(solution):
    def g1(x): #constraint for rosenbrock
        return x[0]**2 + x[1]**2 - 2
    def violate1(x): 
        return 0 if x <= 0 else x
    f1 = rosenbrock(solution)
    f1 += violate1(g1(solution))**2
    return f1

def single_obj_m(solution):
    def g2(x): #constraint for mishra
        return (x[0]+5)**2 + (x[1]+5)**2 - 25
    def violate2(x):
        return 0 if x < 0 else x
    f2 = mishra(solution)
    f2 += violate2(g2(solution))**2
    return f2

multi_problem = {
    "obj_func": multi_obj,
    "bounds": FloatVar(lb=[-10, -6.5], ub=[1.5, 1.5]),
    "minmax": "min",
    "save_population": True,
    "log_to" : None,
}

single_problem_r = {
    "obj_func": single_obj_r,
    "bounds": FloatVar(lb=[-1.5, -1.5], ub=[1.5, 1.5]),
    "minmax": "min", 
    "save_population": True,
    "log_to": None,
}

single_problem_m = {
    "obj_func": single_obj_m,
    "bounds": FloatVar(lb=[-10, -6.5], ub=[0, 0]),
    "minmax": "min", 
    "save_population": True,
    "log_to": None,
}

ga_model = GA.BaseGA(epoch=100, pop_size=50, pc=0.85, pm=0.1)
ga_model.solve(multi_problem)
ga_model.history.save_trajectory_chart(title="trajectory for multi objective search", filename="Examples/trajectory_multi")

In [56]:
multi_solution = ga_model.g_best.solution
multi_fitness = ga_model.g_best.target.fitness
print("Multi_objective solution: ", multi_solution)
print("Multi_objective fitness: ", multi_fitness)

ga_model.solve(single_problem_r)
ga_model.history.save_trajectory_chart(title="trajectory for rosenbrock", filename="Examples/trajectory_rosenbrock")

single_solution_r = ga_model.g_best.solution
single_fitness_r = ga_model.g_best.target.fitness
print("Single objective solution for rosenbrock: ", single_solution_r)
print("Single objective fitness for rosenbrock: ", single_fitness_r)

ga_model.solve(single_problem_m)
ga_model.history.save_trajectory_chart(title="trajectory for mishra", filename="Examples/trajectory_mishra")

single_solution_m = ga_model.g_best.solution
single_fitness_m = ga_model.g_best.target.fitness
print("Single objective solution for mishra: ", single_solution_m)
print("Single objective fitness for mishra: ", single_fitness_m)

score = (abs(rosenbrock(multi_solution) - single_fitness_r) + abs(mishra(multi_solution) - single_fitness_m))

print("Quality score:", score)

Multi_objective solution:  [-0.58775623 -0.66716088]
Multi_objective fitness:  291.199836283631
Single objective solution for rosenbrock:  [0.41334883 0.16515456]
Single objective fitness for rosenbrock:  0.3474116703599565
Single objective solution for mishra:  [-3.13018974 -1.56783326]
Single objective fitness for mishra:  -106.73639827987341
Quality score: 222.2544159528263


In [57]:
def best_trajectory(model):
    trajectory = []

    for agent in model.history.list_population:
        best = min(agent, key=lambda a: a.target.fitness)
        trajectory.append([best.solution[0], best.solution[1]])
    
    return np.array(trajectory)

In [58]:
import matplotlib.pyplot as plt

trajectory = best_trajectory(ga_model)

plt.figure(figsize=(6, 5))
plt.plot(trajectory[:, 0], trajectory[:, 1], marker="o", markersize=3)

plt.title("Best agent trajectory (GA)")
plt.xlabel("Dimension 0")
plt.ylabel("Dimension 1")
plt.grid()

plt.savefig("Examples/GA_best_trajectory_rosenbrock.png")
plt.close()

In [ ]:
from pathlib import Path

charts = ["save_global_objectives_chart", "save_local_objectives_chart", "save_global_best_fitness_chart", 
            "save_local_best_fitness_chart", "save_runtime_chart", "save_exploration_exploitation_chart",
            "save_diversity_chart", "save_trajectory_chart"]

def gen_all_charts(model, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    for chart in charts:
        sliced = chart.replace("save_", "").replace("_chart", "")
        getattr(model.history, chart)(title=sliced, filename=str(out_dir / f"{sliced}.png"))


In [61]:
gen_all_charts(ga_model, "Examples")